# 패키지 및 데이터

In [13]:
# ==================================================
# 기본 라이브러리
# ==================================================
import numpy as np
import seaborn as sb
import pandas as pd

from matplotlib import pyplot as plt
from pandas import DataFrame, concat



# ==================================================
# scikit-learn 공통
# ==================================================
from sklearn.pipeline import Pipeline

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    learning_curve,
)

from sklearn.preprocessing import StandardScaler


# ==================================================
# 분류 모델
# ==================================================
from sklearn.linear_model import (
    LogisticRegression,
    SGDClassifier,
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    log_loss,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score,
)

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
)


# ==================================================
# Boosting 계열
# ==================================================
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


# ==================================================
# 사용자 정의 모듈
# ==================================================
from hossam import *


### 데이터

In [14]:
origin = pd.read_csv("CO2 emission by countries.csv", encoding="latin1")
origin.head()
origin.info()
origin.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59620 entries, 0 to 59619
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Country              59620 non-null  object 
 1   Code                 57452 non-null  object 
 2   Calling Code         56097 non-null  object 
 3   Year                 59620 non-null  int64  
 4   CO2 emission (Tons)  59620 non-null  float64
 5   Population(2022)     53116 non-null  float64
 6   Area                 55284 non-null  float64
 7   % of World           55284 non-null  object 
 8   Density(km2)         53116 non-null  object 
dtypes: float64(3), int64(1), object(5)
memory usage: 4.1+ MB


(59620, 9)

In [15]:
origin.isna().sum()


before = origin.shape[0]

df2 = origin.copy()
df2.head()
d = df2.dropna()

# DataFrame() 은 리스트/배열/Series 형태가 필요합니다.
after = d.shape[0]

# before = origin.shape[0]
# after  = d.shape[0] 은 정수(int)이기 때문에 .mean()메서드 적용 불가
# 반드시 수치형으로 변환

before_mean = origin.mean(numeric_only=True)
after_mean = d.mean(numeric_only=True)
od = pd.DataFrame(
    {
        "결측치 제거 전":[before],
        "결측치 제거 후": [after],
        "결측치 제거 전 평균값": before_mean.mean(),
        "결측치 제거 후 평균값": after_mean.mean()
       
    }
)

od




,결측치 제거 전,결측치 제거 후,결측치 제거 전 평균값,결측치 제거 후 평균값
0,59620,48509,268837596.160,300811233.042


### 행 개수 + 평균 한 번에 정리

In [16]:
summary = pd.DataFrame({
    "rows": [origin.shape[0], d.shape[0]],
}, index=["before_dropna", "after_dropna"])

mean_compare = pd.DataFrame({
    "before_mean": origin.mean(numeric_only=True),
    "after_mean": d.mean(numeric_only=True)
})

summary
mean_compare


,before_mean,after_mean
Year,1885.000,1885.000
CO2 emission (Tons),1034773694.740,1160748250.497
Population(2022),39922597.597,41861989.536
Area,652207.304,632807.134


### 분석 관점에서 더 좋은 비교

In [17]:
pd.concat(
    [
        origin.mean(numeric_only=True),
        d.mean(numeric_only=True)
    ],
    axis=1,
    keys=["Before", "After"]
)
 

,Before,After
Year,1885.000,1885.000
CO2 emission (Tons),1034773694.740,1160748250.497
Population(2022),39922597.597,41861989.536
Area,652207.304,632807.134


### 평균 변화량 계산 코드

In [18]:
mean_compare = pd.concat(
    [
        origin.mean(numeric_only=True),
        d.mean(numeric_only=True)
    ],
    axis=1
)

mean_compare.columns = ["Before", "After"]
mean_compare["Difference"] = (
    mean_compare["After"] - mean_compare["Before"]
)

mean_compare


,Before,After,Difference
Year,1885.000,1885.000,0.000
CO2 emission (Tons),1034773694.740,1160748250.497,125974555.757
Population(2022),39922597.597,41861989.536,1939391.939
Area,652207.304,632807.134,-19400.170


### 어떤 칼럼이 가장 많이 줄었는지 자동 탐색

In [19]:
# 칼럼별 결측 개수 비교
na_compare = pd.DataFrame({
    "결측 전 NA 개수": origin.isna().sum(),
    "결측 전 NA 비율(%)": origin.isna().mean() * 100
}).sort_values("결측 전 NA 비율(%)", ascending=False)

na_compare


,결측 전 NA 개수,결측 전 NA 비율(%)
Population(2022),6504,10.909
Density(km2),6504,10.909
Area,4336,7.273
% of World,4336,7.273
Calling Code,3523,5.909
Code,2168,3.636
Country,0,0.000
Year,0,0.000
CO2 emission (Tons),0,0.000


### 데이터 전처리

In [23]:
# 다른 데이터프레임으로 reset_index()
df = origin.copy()
df.reset_index().rename(columns = {"index":"Hello"})

,Hello,Country,Code,Calling Code,Year,CO2 emission (Tons),Population(2022),Area,% of World,Density(km2)
0,0,Afghanistan,AF,93,1750,0.000,41128771.000,652230.000,0.40%,63/km²
1,1,Afghanistan,AF,93,1751,0.000,41128771.000,652230.000,0.40%,63/km²
2,2,Afghanistan,AF,93,1752,0.000,41128771.000,652230.000,0.40%,63/km²
3,3,Afghanistan,AF,93,1753,0.000,41128771.000,652230.000,0.40%,63/km²
4,4,Afghanistan,AF,93,1754,0.000,41128771.000,652230.000,0.40%,63/km²
...,...,...,...,...,...,...,...,...,...,...
59615,59615,Zimbabwe,ZW,263,2016,736467042.000,16320537.000,390757.000,0.30%,42/km²
59616,59616,Zimbabwe,ZW,263,2017,746048675.000,16320537.000,390757.000,0.30%,42/km²
59617,59617,Zimbabwe,ZW,263,2018,757903042.000,16320537.000,390757.000,0.30%,42/km²
59618,59618,Zimbabwe,ZW,263,2019,768852126.000,16320537.000,390757.000,0.30%,42/km²


In [28]:
#reset_index()를 통해 항목 계산, 칼럼 하나가 더 생김
df1 = df['Country'].value_counts().reset_index()

df1.columns = ['Country', 'COUNT']

df1

,Country,COUNT
0,Afghanistan,271
1,Palestine,271
2,New Zealand,271
3,Nicaragua,271
4,Niger,271
...,...,...
215,Greece,271
216,Greenland,271
217,Grenada,271
218,Guadeloupe,271


In [ ]:
# pd.concat()공부


### 2개의 연속형 칼럼 CO2 emission (Tons) x Population(2022) 계산